#Assignment #6

Vision Transformer

---


**SUBMISSION INSTRUCTIONS**

First make a copy of this colab file and then solve the assignment and upload your final notebook on github.

Before uploading your downloaded notebook, RENAME the file as rollno_name.ipynb

Submission Deadline : 31/01/2026 Saturday EOD i.e before 11:59 PM

The deadline is strict, Late submissions will incur penalty

Note that you have to upload your solution on the github page of the project Vision Transformer and **under Assignment 6**

And remember to keep title of your pull request to be ViT_name_rollno_assgn6

Github Submission repo -
https://github.com/electricalengineersiitk/Winter-projects-25-26/tree/main/Vision%20transformer/Assignment%206

# Part A - Data and tokens

Q1. Build a tiny toy dataset with pandas
Create a pandas DataFrame with columns text and label.
- Include at least 12 short sentences (3-10 words each).
- The label is 0/1 (e.g., positive vs negative sentiment).
- Shuffle rows and split into train/test (80/20) using a fixed random seed.
Return: df_train, df_test.

In [1]:
import pandas as pd

def make_toy_dataset(seed: int = 42):
    """Return df_train, df_test with columns: text (str), label (int)."""

    # Create dataset (12+ sentences, 3–10 words each)
    data = [
        ("i love this movie", 1),
        ("this film was amazing", 1),
        ("what a great experience", 1),
        ("i really liked this", 1),
        ("absolutely fantastic story", 1),
        ("this was a good movie", 1),
        ("i hate this movie", 0),
        ("this film was terrible", 0),
        ("what a bad experience", 0),
        ("i really disliked this", 0),
        ("absolutely boring story", 0),
        ("this was a bad movie", 0),
    ]

    df = pd.DataFrame(data, columns=["text", "label"])

    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    split_idx = int(0.8 * len(df))
    df_train = df.iloc[:split_idx].reset_index(drop=True)
    df_test = df.iloc[split_idx:].reset_index(drop=True)

    return df_train, df_test

Q2. Clean and tokenize text

Implement a basic cleaner: lowercase, strip, replace multiple spaces with one, and remove punctuation
(.,!?;:).
Tokenize by whitespace.
Add a new column tokens that stores a list of tokens per row.
Return the updated DataFrame.

In [2]:
import re
import pandas as pd

def clean_text(s: str) -> str:
    s = s.lower().strip()

    s = re.sub(r'[.,!?;:]', '', s)

    s = re.sub(r'\s+', ' ', s)

    return s


def add_tokens_column(df: pd.DataFrame) -> pd.DataFrame:

    df['text'] = df['text'].apply(clean_text)

    df['tokens'] = df['text'].apply(lambda x: x.split())

    return df

Q3. Build a vocabulary + token/id mappings

Build token2id and id2token using the training tokens.
Include special tokens: [PAD], [UNK], [BOS], [EOS] at the beginning.
Add tokens that occur at least min_freq times.
Return: token2id (dict), id2token (list).

In [3]:
from collections import Counter

SPECIALS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]']

def build_vocab(list_of_token_lists, min_freq: int = 1):
    counter = Counter()
    for tokens in list_of_token_lists:
        counter.update(tokens)
    vocab = SPECIALS + [t for t, f in counter.items() if f >= min_freq]
    token2id = {t: i for i, t in enumerate(vocab)}
    id2token = vocab
    return token2id, id2token

Q4. Convert tokens to ids + pad to a batch

Implement tokens_to_ids for one sequence.
Implement pad_batch that takes a list of id sequences and returns:
- X: int array (B,T) padded with pad_id
- pad_mask: bool array (B,T) where True means 'real token' and False means padding

In [4]:
import numpy as np

def tokens_to_ids(tokens, token2id, unk_token='[UNK]'):
    return [token2id.get(t, token2id[unk_token]) for t in tokens]

def pad_batch(id_seqs, pad_id: int):
    B = len(id_seqs)
    T = max(len(s) for s in id_seqs)
    X = np.full((B, T), pad_id)
    pad_mask = np.zeros((B, T), dtype=bool)
    for i, s in enumerate(id_seqs):
        X[i, :len(s)] = s
        pad_mask[i, :len(s)] = True
    return X, pad_mask

#Part B - Core Transformer math

Q5. Embedding lookup

Implement an embedding table E of shape (V,D) initialized from a normal distribution (mean 0, std 0.02).
Given token ids X (B,T), return embeddings of shape (B,T,D) using NumPy indexing.


In [5]:
import numpy as np

def init_embeddings(vocab_size: int, d_model: int, seed: int = 0):
    np.random.seed(seed)
    return np.random.normal(0, 0.02, (vocab_size, d_model))

def embed(X: np.ndarray, E: np.ndarray):
    return E[X]

Q6. Sinusoidal positional encoding

Implement the classic sinusoidal positional encoding PE of shape (T,D).
Then add it to token embeddings (B,T,D).
Make sure your implementation works for both even and odd D.

In [8]:
import numpy as np

def sinusoidal_positional_encoding(T: int, D: int):
    PE = np.zeros((T, D))
    for pos in range(T):
        for i in range(0, D, 2):
            PE[pos, i] = np.sin(pos / (10000 ** (i / D)))
            if i + 1 < D:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / D)))
    return PE

def add_positional_encoding(X_emb: np.ndarray, PE: np.ndarray):
    return X_emb + PE

Q7. Scaled dot-product attention with masking

Implement scaled dot-product attention:
Attention(Q,K,V) = softmax((Q @ K^T) / sqrt(dk) + mask) @ V
Inputs: Q,K,V are (B,H,T,Dh). Mask is boolean broadcastable to (B,H,T,T) where False means 'mask out'.
Requirements:
- Use a numerically stable softmax (subtract max).
- Convert boolean mask to large negative values before softmax.
Return: context (B,H,T,Dh) and attention weights (B,H,T,T).

In [9]:
import numpy as np

def softmax(x: np.ndarray, axis: int = -1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    dk = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0,1,3,2)) / np.sqrt(dk)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    attn = softmax(scores, axis=-1)
    context = np.matmul(attn, V)
    return context, attn

Q8. Multi-head self-attention (MHA)

Implement multi-head self-attention for input X (B,T,D).
- Project to Q,K,V using weight matrices Wq,Wk,Wv each (D,D).
- Reshape/split into heads -> (B,H,T,Dh) where Dh=D/H.
- Apply scaled dot-product attention with a pad mask (B,T) (broadcast it appropriately).
- Concatenate heads and apply output projection Wo (D,D).
Return: out (B,T,D) and attention weights (B,H,T,T).

In [10]:
import numpy as np

def linear(x: np.ndarray, W: np.ndarray, b=None):
    y = x @ W
    if b is not None:
        y += b
    return y

def split_heads(x: np.ndarray, n_heads: int):
    B, T, D = x.shape
    Dh = D // n_heads
    return x.reshape(B, T, n_heads, Dh).transpose(0, 2, 1, 3)

def combine_heads(xh: np.ndarray):
    B, H, T, Dh = xh.shape
    return xh.transpose(0, 2, 1, 3).reshape(B, T, H * Dh)

def mha_self_attention(X, Wq, Wk, Wv, Wo, n_heads: int, pad_mask=None):
    Q = linear(X, Wq)
    K = linear(X, Wk)
    V = linear(X, Wv)

    Qh = split_heads(Q, n_heads)
    Kh = split_heads(K, n_heads)
    Vh = split_heads(V, n_heads)

    mask = pad_mask[:, None, None, :] if pad_mask is not None else None

    context, attn = scaled_dot_product_attention(Qh, Kh, Vh, mask)

    out = combine_heads(context)
    out = linear(out, Wo)

    return out, attn

Q9. LayerNorm + residual connection

Implement LayerNorm for X (B,T,D) using learnable gamma and beta of shape (D,).
Then implement residual_add_and_norm(Y, X, gamma, beta) that returns LayerNorm(X + Y).

In [11]:
import numpy as np

def layer_norm(X: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-5):
    mean = X.mean(axis=-1, keepdims=True)
    var = X.var(axis=-1, keepdims=True)
    Xn = (X - mean) / np.sqrt(var + eps)
    return gamma * Xn + beta

def residual_add_and_norm(Y: np.ndarray, X: np.ndarray, gamma: np.ndarray, beta: np.ndarray):
    return layer_norm(X + Y, gamma, beta)

Q10. Position-wise FeedForward network

Implement FFN: FFN(X) = relu(X @ W1 + b1) @ W2 + b2
Shapes: X (B,T,D), W1 (D,Dff), b1 (Dff,), W2 (Dff,D), b2 (D,)
Return: (B,T,D).

In [12]:
import numpy as np

def relu(x: np.ndarray):
    return np.maximum(0, x)

def feed_forward(X: np.ndarray, W1: np.ndarray, b1: np.ndarray, W2: np.ndarray, b2: np.ndarray):
    return relu(X @ W1 + b1) @ W2 + b2

# Part C - Putting it together

Q11. One Transformer encoder block (forward)

Implement a single encoder block forward pass:
1) MHA = MultiHeadSelfAttention(X) with pad_mask
2) X1 = LayerNorm(X + MHA)
3) FFN = FeedForward(X1)
4) X2 = LayerNorm(X1 + FFN)
Return X2.
You may pass all parameters explicitly (weights, gamma/beta).

In [13]:
def encoder_block_forward(X, params, n_heads: int, pad_mask=None):
    mha_out, _ = mha_self_attention(X, params['Wq'], params['Wk'], params['Wv'], params['Wo'], n_heads, pad_mask)
    X1 = residual_add_and_norm(mha_out, X, params['gamma1'], params['beta1'])
    ffn_out = feed_forward(X1, params['W1'], params['b1'], params['W2'], params['b2'])
    X2 = residual_add_and_norm(ffn_out, X1, params['gamma2'], params['beta2'])
    return X2

Q12. Sequence classification head + end-to-end demo

Create an end-to-end forward pass for a tiny classifier:
- Input ids -> embeddings + positional enc
- One encoder block
- Pooling: take the [BOS] position (t=0) as the sequence representation
- Linear head: logits = h0 @ Wcls + bcls with Wcls (D,2), bcls (2,)
- Softmax to probabilities
Write predict_proba that takes a batch of texts and returns probs (B,2).
Include simple sanity checks: shapes, probabilities sum to 1, and masking doesn't crash for different
lengths.


In [14]:
def predict_proba(texts, token2id, E, PE, params, Wcls, bcls, n_heads: int):
    tokenized = [[token2id.get(w, token2id['[UNK]']) for w in t.split()] for t in texts]
    X, mask = pad_batch(tokenized, token2id['[PAD]'])
    X_emb = embed(X, E)
    X_emb = add_positional_encoding(X_emb, PE[:X.shape[1]])
    X_enc = encoder_block_forward(X_emb, params, n_heads, mask)
    h0 = X_enc[:, 0, :]
    logits = h0 @ Wcls + bcls
    probs = softmax(logits, axis=-1)
    return probs